# Manhwa Studio — Google Colab + Google Drive

Versão independente para renderizar com FFmpeg no ambiente do Colab e ler/gravar arquivos diretamente no Google Drive. A interface web original continua separada.

**Uso autorizado:** processe somente vídeos que você possui ou tem autorização para baixar e editar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK = Path('/content/drive/MyDrive/ManhwaStudio')
INPUT = WORK / 'entrada'
OUTPUT = WORK / 'exportados'
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Entrada:', INPUT)
print('Saída:', OUTPUT)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!ffmpeg -version | head -n 2
!nvidia-smi || true

In [ ]:
!git clone --depth 1 --branch google-colab-drive https://github.com/manhwavaz1-lab/manhwa-recreator-studio.git /content/manhwa-recreator-studio
%cd /content/manhwa-recreator-studio
!python -m pip install -q yt-dlp

## 1. Escolha os arquivos no Drive
Coloque o vídeo MP4 e a moldura PNG/JPG na pasta `ManhwaStudio/entrada`. Se preferir, informe um link do YouTube no próximo bloco.

In [ ]:
from pathlib import Path
videos = sorted([*INPUT.glob('*.mp4'), *INPUT.glob('*.mov'), *INPUT.glob('*.mkv'), *INPUT.glob('*.webm')])
frames = sorted([*INPUT.glob('*.png'), *INPUT.glob('*.jpg'), *INPUT.glob('*.jpeg')])
print('Vídeos:', [p.name for p in videos])
print('Molduras:', [p.name for p in frames])

# Edite estes caminhos se houver mais de um arquivo:
VIDEO = str(videos[0]) if videos else ''
FRAME = str(frames[0]) if frames else ''
print('VIDEO =', VIDEO, '\\nFRAME =', FRAME)

## 2. Opcional: baixar vídeo do YouTube
Use apenas quando tiver direito/autorização para baixar o conteúdo. Deixe `YOUTUBE_URL` vazio se já colocou o MP4 no Drive.

In [ ]:
YOUTUBE_URL = ''  # cole aqui o link, se necessário
if YOUTUBE_URL.strip():
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'yt_dlp', '-f', 'bv*+ba/b', '--merge-output-format', 'mp4', '-o', str(INPUT / 'youtube_source.%(ext)s'), YOUTUBE_URL], check=True)
    downloaded = sorted(INPUT.glob('youtube_source.*'), key=lambda p:p.stat().st_mtime, reverse=True)
    VIDEO = str(downloaded[0])
print('Vídeo selecionado:', VIDEO or 'Nenhum')

## 3. Configurar e renderizar
Padrão recomendado: 1080p, 30 FPS, H.264. O script tenta NVENC/QSV/AMF quando o FFmpeg disponibiliza o encoder e retorna automaticamente para CPU (`libx264`) se nenhum encoder de hardware estiver disponível. Colab pode oferecer NVIDIA, mas a GPU depende do tipo de runtime atribuído.

In [ ]:
import subprocess, sys
assert VIDEO and Path(VIDEO).exists(), 'Defina VIDEO para um arquivo existente.'
assert FRAME and Path(FRAME).exists(), 'Defina FRAME para uma moldura PNG/JPG existente.'
OUT = str(OUTPUT / 'manhwa-studio-export.mp4')

FORMAT = '16:9'       # ou '9:16'
ENCODER = 'auto'      # auto, nvenc, qsv, amf ou cpu
FPS = 30
ZOOM = 100            # 50 a 250
CROP_TO_FILL = True   # True recorta para preencher; False preserva o vídeo inteiro
OFFSET_X = 0          # -50 a 50
OFFSET_Y = 0          # -50 a 50
BLUR_REGION = None    # ex.: '80,85,15,8' (x,y,largura,altura em % do vídeo original)

cmd=[sys.executable, 'colab/renderer.py', '--video', VIDEO, '--frame', FRAME, '--output', OUT,
     '--format', FORMAT, '--encoder', ENCODER, '--fps', str(FPS), '--zoom', str(ZOOM),
     '--offset-x', str(OFFSET_X), '--offset-y', str(OFFSET_Y)]
if CROP_TO_FILL: cmd.append('--crop')
if BLUR_REGION: cmd += ['--blur', BLUR_REGION]
subprocess.run(cmd, check=True)
print('Exportado para o Google Drive:', OUT)

In [ ]:
from IPython.display import Video, display
display(Video(OUT, embed=False))
print('Arquivo salvo em:', OUT)